# 01 · Bronze — Ingesta AEMET (climatología + DANA)

 Este notebook llama a la API de AEMET OpenData en dos frentes:
 1. **Climatología histórica** de estaciones representativas de España (temperatura + precipitación)
2. **Episodio DANA Valencia** (28 oct - 5 nov 2024) en estaciones de la zona afectada
 La API de AEMET funciona en **dos pasos**: la primera llamada devuelve una URL temporal, la segunda llamada a esa URL trae los datos reales. Además tiene límites de rango por petición, así que troceamos las fechas.
La API key se recupera del Secret Scope `aemet` — nunca aparece en texto plano en este notebook.

## 0. Setup




In [0]:
import requests
import time
import json
from datetime import datetime, timedelta

dbutils.widgets.text("catalog_name", "global_climate", "Catálogo")
dbutils.widgets.text("target_cities", "MADRID,BARCELONA,VALENCIA,SEVILLA,BILBAO", "Ciudades objetivo (climatología)")
dbutils.widgets.text("years_back", "15", "Años hacia atrás (climatología)")
dbutils.widgets.text("dana_start", "2024-10-28", "Inicio DANA")
dbutils.widgets.text("dana_end", "2024-11-05", "Fin DANA")

catalog_name = dbutils.widgets.get("catalog_name")
target_cities = [c.strip().upper() for c in dbutils.widgets.get("target_cities").split(",")]
years_back = int(dbutils.widgets.get("years_back"))
dana_start = dbutils.widgets.get("dana_start")
dana_end = dbutils.widgets.get("dana_end")

API_BASE = "https://opendata.aemet.es/opendata/api"
API_KEY = dbutils.secrets.get(scope="aemet", key="api_key")

ingest_date = datetime.now().strftime("%Y-%m-%d")
print(f"Catálogo: {catalog_name} | Ciudades: {target_cities} | Años atrás: {years_back}")
print(f"Ventana DANA: {dana_start} → {dana_end} | Fecha de ingesta: {ingest_date}")


## 1. Funciones auxiliares

 - `call_aemet`: hace la llamada en dos pasos, con reintento si hay error 429 (demasiadas peticiones)
- `chunk_date_range`: trocea un rango de fechas en tramos de ~150 días (el límite por estación son 6 meses, dejamos margen)



In [0]:

def call_aemet(endpoint, max_retries=3):
    """Llama a un endpoint de AEMET siguiendo el patrón de dos pasos. Devuelve la lista de registros o None."""
    url = f"{API_BASE}/{endpoint}"
    headers = {"api_key": API_KEY, "cache-control": "no-cache"}

    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers)

        if resp.status_code == 429:
            wait = 5 * (attempt + 1)
            print(f"  Rate limit alcanzado, esperando {wait}s...")
            time.sleep(wait)
            continue

        if resp.status_code != 200:
            print(f"  Error HTTP {resp.status_code} en {endpoint}")
            return None

        payload = resp.json()
        if payload.get("estado") != 200:
            print(f"  AEMET devolvió estado {payload.get('estado')}: {payload.get('descripcion')}")
            return None

        # Segunda llamada: la URL temporal con los datos reales
        data_resp = requests.get(payload["datos"])
        if data_resp.status_code != 200:
            print(f"  Error al recuperar datos reales: HTTP {data_resp.status_code}")
            return None

        return data_resp.json()

    print(f"  Máximo de reintentos alcanzado en {endpoint}")
    return None


def chunk_date_range(start_date, end_date, chunk_days=150):
    """Genera tuplas (inicio, fin) en formato AEMET (fechaTXX:00:00UTC) sin superar chunk_days por tramo."""
    chunks = []
    current = start_date
    while current < end_date:
        chunk_end = min(current + timedelta(days=chunk_days), end_date)
        chunks.append((current, chunk_end))
        current = chunk_end + timedelta(days=1)
    return chunks


def format_aemet_date(d, end_of_day=False):
    suffix = "T23:59:59UTC" if end_of_day else "T00:00:00UTC"
    return d.strftime("%Y-%m-%d") + suffix



## 2. Inventario de estaciones — localizar códigos por ciudad



In [0]:
print("Descargando inventario completo de estaciones...")
inventario = call_aemet("valores/climatologicos/inventarioestaciones/todasestaciones")

if inventario is None:
    raise RuntimeError("No se pudo descargar el inventario de estaciones. Revisa la API key y el estado del servicio.")

print(f"Total de estaciones en el inventario: {len(inventario)}")

# Filtramos: una estación por cada ciudad objetivo + cualquier estación de Chiva (zona DANA)
estaciones_climatologia = {}
for ciudad in target_cities:
    candidatas = [e for e in inventario if ciudad in e.get("nombre", "").upper()]
    if candidatas:
        estaciones_climatologia[ciudad] = candidatas[0]["indicativo"]
        print(f"  {ciudad} → {candidatas[0]['indicativo']} ({candidatas[0]['nombre']})")
    else:
        print(f"  ⚠ No se encontró estación para {ciudad}")

estaciones_dana = [e for e in inventario if "CHIVA" in e.get("nombre", "").upper() or "VALENCIA" in e.get("nombre", "").upper()]
print(f"\nEstaciones candidatas para la DANA: {[(e['indicativo'], e['nombre']) for e in estaciones_dana]}")


## 3. Ingesta climatología histórica
Recorre cada estación objetivo en tramos de fechas, respetando el límite de la API (pausa entre llamadas para no superar el rate limit de 50 peticiones/minuto).



In [0]:
end_date = datetime.now()
start_date = end_date.replace(year=end_date.year - years_back)
date_chunks = chunk_date_range(start_date, end_date)

print(f"Rango total: {start_date.date()} → {end_date.date()} en {len(date_chunks)} tramos por estación")
print(f"Peticiones estimadas: {len(date_chunks) * len(estaciones_climatologia)}")

registros_climatologia = []

for ciudad, indicativo in estaciones_climatologia.items():
    print(f"\n--- {ciudad} ({indicativo}) ---")
    for i, (ini, fin) in enumerate(date_chunks):
        endpoint = (
            f"valores/climatologicos/diarios/datos/"
            f"fechaini/{format_aemet_date(ini)}/"
            f"fechafin/{format_aemet_date(fin, end_of_day=True)}/"
            f"estacion/{indicativo}"
        )
        resultado = call_aemet(endpoint)
        if resultado:
            for r in resultado:
                r["ciudad_objetivo"] = ciudad
            registros_climatologia.extend(resultado)
            print(f"  Tramo {i+1}/{len(date_chunks)}: {len(resultado)} registros")
        time.sleep(1.5)  # margen de seguridad frente al rate limit

print(f"\nTotal registros de climatología descargados: {len(registros_climatologia)}")



## 4. Ingesta específica del episodio DANA



In [0]:
dana_start_dt = datetime.strptime(dana_start, "%Y-%m-%d")
dana_end_dt = datetime.strptime(dana_end, "%Y-%m-%d")

registros_dana = []

for est in estaciones_dana:
    indicativo = est["indicativo"]
    endpoint = (
        f"valores/climatologicos/diarios/datos/"
        f"fechaini/{format_aemet_date(dana_start_dt)}/"
        f"fechafin/{format_aemet_date(dana_end_dt, end_of_day=True)}/"
        f"estacion/{indicativo}"
    )
    resultado = call_aemet(endpoint)
    if resultado:
        registros_dana.extend(resultado)
        print(f"  {est['nombre']} ({indicativo}): {len(resultado)} registros")
    time.sleep(1.5)

print(f"\nTotal registros DANA descargados: {len(registros_dana)}")



## 5. Aterrizaje en el Volume (JSON crudo, tal cual llega de la API)


In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.bronze.aemet_climatologia")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.bronze.aemet_dana")

vol_climatologia = f"/Volumes/{catalog_name}/bronze/aemet_climatologia/ingest_date={ingest_date}"
vol_dana = f"/Volumes/{catalog_name}/bronze/aemet_dana/ingest_date={ingest_date}"

dbutils.fs.mkdirs(vol_climatologia)
dbutils.fs.mkdirs(vol_dana)

dbutils.fs.put(f"{vol_climatologia}/climatologia_raw.json", json.dumps(registros_climatologia), overwrite=True)
dbutils.fs.put(f"{vol_dana}/dana_raw.json", json.dumps(registros_dana), overwrite=True)

print(f"Guardado en: {vol_climatologia}/climatologia_raw.json")
print(f"Guardado en: {vol_dana}/dana_raw.json")



## 6. Carga como tabla Delta en `bronze`


In [0]:
df_climatologia = spark.read.json(f"{vol_climatologia}/climatologia_raw.json")
df_climatologia.write.mode("overwrite").saveAsTable(f"{catalog_name}.bronze.aemet_climatologia_raw")

df_dana = spark.read.json(f"{vol_dana}/dana_raw.json")
df_dana.write.mode("overwrite").saveAsTable(f"{catalog_name}.bronze.aemet_dana_raw")

print("Tablas Delta creadas:")
print(f"  - {catalog_name}.bronze.aemet_climatologia_raw")
print(f"  - {catalog_name}.bronze.aemet_dana_raw")


## 7. Verificación


In [0]:
print("=== Muestra climatología ===")
display(df_climatologia.limit(10))

print("=== Muestra DANA ===")
display(df_dana.limit(10))



## 8. Siguiente paso
Con `aemet_climatologia_raw` y `aemet_dana_raw` ya en bronze, el siguiente notebook (`02_silver/clean_temperature.py`) se encargará de tipar los campos (AEMET devuelve números como texto con coma decimal, ej. `"18,4"`), unificar unidades y calcular las primeras features (medias móviles, anomalías por estación).